# 07 — Visualisation: Attention Maps, Grad-CAM & Segmentation Overlays
**BraTS 2023 GLI Brain Tumor Segmentation | RCOEM 2026–27**

This notebook generates all qualitative visualisations for the project report:
1. **Predicted segmentation overlays** on MRI slices — Baseline vs Attention U-Net
2. **Spatial attention maps** from all 4 decoder stages (interpretability)
3. **Grad-CAM heatmaps** showing what regions drive predictions
4. **Error analysis** — false positives / false negatives visualised
5. **Demo figure** — publication-quality side-by-side comparison

**Estimated time:** ~30 minutes

In [ ]:
import sys, os
DATASET_PATH = '/home/yourname/BraTS2023_Training_Data'  # ← CHANGE THIS
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

from src.models  import UNet3D, AttentionUNet3D
from src.dataset import get_patient_folders, load_patient, get_file_paths
from src.utils   import load_checkpoint, sliding_window_inference
from src.metrics import compute_patient_metrics
from src.config  import DEVICE, PATCH_SIZE, PATCH_OVERLAP, MAX_PATIENTS, RANDOM_SEED

os.makedirs('results/figures', exist_ok=True)
print(f'Device: {DEVICE}')
print('✅ Ready for visualisation')

## 1️⃣ Load Models

In [ ]:
baseline_model = UNet3D(4, 4, 32).to(DEVICE)
load_checkpoint(baseline_model, model_name='baseline_unet3d', prefer_best=True)
baseline_model.eval()

attn_model = AttentionUNet3D(4, 4, 32, use_channel_attn=True, use_spatial_attn=True).to(DEVICE)
load_checkpoint(attn_model, model_name='attention_unet3d_full', prefer_best=True)
attn_model.eval()

print('✅ Both models loaded (best checkpoints)')

## 2️⃣ Pick Test Patients

In [ ]:
np.random.seed(RANDOM_SEED)
all_folders = get_patient_folders(DATASET_PATH)
if MAX_PATIENTS:
    all_folders = all_folders[:MAX_PATIENTS]
np.random.shuffle(all_folders)
n_test = max(1, int(len(all_folders) * 0.10))
test_folders = all_folders[-n_test:]

# Choose 3 test patients for detailed visualisation
VIZ_FOLDERS = test_folders[:3]
print(f'Visualising {len(VIZ_FOLDERS)} patients:')
for f in VIZ_FOLDERS:
    print(f'  {Path(f).name}')

## 3️⃣ Segmentation Overlay — Baseline vs Attention U-Net

In [ ]:
def overlay_seg_on_mri(mri_slice, seg_slice, alpha=0.5):
    """Return RGBA overlay of segmentation on MRI slice."""
    # Normalise MRI to 0-1
    mri_norm = (mri_slice - mri_slice.min()) / (mri_slice.max() - mri_slice.min() + 1e-8)
    rgb = np.stack([mri_norm]*3, axis=-1)  # grayscale → RGB

    label_colors_rgb = {
        1: np.array([0.91, 0.30, 0.24]),  # NCR  = red
        2: np.array([0.20, 0.60, 0.86]),  # ED   = blue
        3: np.array([0.18, 0.80, 0.44]),  # ET   = green
    }
    for label_id, color in label_colors_rgb.items():
        mask = seg_slice == label_id
        rgb[mask] = (1 - alpha) * rgb[mask] + alpha * color
    return rgb


for folder in VIZ_FOLDERS:
    patient_id = Path(folder).name
    image, gt_seg = load_patient(folder, has_seg=True)
    img_tensor = torch.from_numpy(image.astype(np.float32))

    with torch.no_grad():
        pred_baseline = sliding_window_inference(baseline_model, img_tensor, PATCH_SIZE, PATCH_OVERLAP, DEVICE)
        pred_attn     = sliding_window_inference(attn_model,     img_tensor, PATCH_SIZE, PATCH_OVERLAP, DEVICE)

    # Compute dice for annotation
    m_baseline = compute_patient_metrics(pred_baseline, gt_seg)
    m_attn     = compute_patient_metrics(pred_attn,     gt_seg)

    flair = image[0]  # FLAIR modality for background
    mid_z = gt_seg.shape[2] // 2

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'Patient: {patient_id}', fontsize=13, fontweight='bold')

    # Col 0: FLAIR
    axes[0].imshow(flair[:, :, mid_z].T, cmap='gray', origin='lower')
    axes[0].set_title('FLAIR (Input)', fontsize=11)
    axes[0].axis('off')

    # Col 1: Ground truth
    gt_overlay = overlay_seg_on_mri(flair[:, :, mid_z], gt_seg[:, :, mid_z])
    axes[1].imshow(gt_overlay.transpose(1, 0, 2), origin='lower')
    axes[1].set_title('Ground Truth', fontsize=11)
    axes[1].axis('off')

    # Col 2: Baseline prediction
    bl_overlay = overlay_seg_on_mri(flair[:, :, mid_z], pred_baseline[:, :, mid_z])
    axes[2].imshow(bl_overlay.transpose(1, 0, 2), origin='lower')
    bl_dice = m_baseline['WT']['dice']
    axes[2].set_title(f'Baseline U-Net\n(WT Dice={bl_dice:.3f})', fontsize=11)
    axes[2].axis('off')

    # Col 3: Attention U-Net prediction
    at_overlay = overlay_seg_on_mri(flair[:, :, mid_z], pred_attn[:, :, mid_z])
    axes[3].imshow(at_overlay.transpose(1, 0, 2), origin='lower')
    at_dice = m_attn['WT']['dice']
    axes[3].set_title(f'Attention U-Net (Ours)\n(WT Dice={at_dice:.3f})', fontsize=11, color='#27ae60')
    axes[3].axis('off')

    # Legend
    patches = [
        mpatches.Patch(color='#e74c3c', label='NCR (Label 1)'),
        mpatches.Patch(color='#3498db', label='Edema (Label 2)'),
        mpatches.Patch(color='#2ecc71', label='ET (Label 3)'),
    ]
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.05))

    plt.tight_layout()
    save_path = f'results/figures/segmentation_overlay_{patient_id}.png'
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

## 4️⃣ Spatial Attention Maps — Where Does the Model Look?

In [ ]:
# Use first test patient for attention map visualisation
folder = VIZ_FOLDERS[0]
patient_id = Path(folder).name
image, gt_seg = load_patient(folder, has_seg=True)

# Extract a single patch around the tumor for attention maps
# (full volume attention map would be too large — use a 96^3 patch)
from src.dataset import extract_random_patch
import torch.nn.functional as F

img_patch, seg_patch = extract_random_patch(image, gt_seg, PATCH_SIZE, foreground_prob=1.0)
img_tensor = torch.from_numpy(img_patch.astype(np.float32)).unsqueeze(0).to(DEVICE)  # (1,4,H,W,D)

with torch.no_grad():
    logits, attention_maps = attn_model.get_attention_maps(img_tensor)

print(f'Number of attention maps: {len(attention_maps)}')
for i, am in enumerate(attention_maps):
    print(f'  Decoder level {i+1}: shape={am.shape}')

# Visualise attention maps
mid = img_patch.shape[-1] // 2  # axial mid-slice of the patch

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle(f'Spatial Attention Maps — {patient_id}\n(Decoder levels 1–4)', fontsize=13, fontweight='bold')

flair_slice = img_patch[0, :, :, mid]
flair_norm  = (flair_slice - flair_slice.min()) / (flair_slice.max() - flair_slice.min() + 1e-8)

# Custom hot colourmap for attention
attn_cmap = plt.cm.hot

for i, attn_map in enumerate(attention_maps):
    attn = attn_map[0, 0].numpy()  # (H', W', D') — squeeze batch & channel

    # Upsample to patch size for overlay
    attn_upsampled = F.interpolate(
        torch.from_numpy(attn).unsqueeze(0).unsqueeze(0).float(),
        size=img_patch.shape[1:], mode='trilinear', align_corners=True
    )[0, 0].numpy()

    attn_slice = attn_upsampled[:, :, mid]
    seg_slice  = seg_patch[:, :, mid]

    # Row 0: FLAIR + attention heatmap overlay
    axes[0, i].imshow(flair_norm.T,    cmap='gray', origin='lower', alpha=1.0)
    axes[0, i].imshow(attn_slice.T,    cmap=attn_cmap, origin='lower', alpha=0.5)
    axes[0, i].set_title(f'Decoder Level {i+1}\n(attention overlay)', fontsize=10)
    axes[0, i].axis('off')

    # Row 1: attention map alone
    im = axes[1, i].imshow(attn_slice.T, cmap=attn_cmap, origin='lower')
    axes[1, i].set_title(f'Attention Map (Level {i+1})\nmin={attn_slice.min():.2f} max={attn_slice.max():.2f}', fontsize=9)
    axes[1, i].axis('off')
    plt.colorbar(im, ax=axes[1, i], fraction=0.04)

plt.tight_layout()
plt.savefig(f'results/figures/attention_maps_{patient_id}.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: results/figures/attention_maps_{patient_id}.png')

## 5️⃣ Error Analysis — False Positives & False Negatives

In [ ]:
folder = VIZ_FOLDERS[0]
image, gt_seg = load_patient(folder, has_seg=True)
img_tensor = torch.from_numpy(image.astype(np.float32))

with torch.no_grad():
    pred_bl   = sliding_window_inference(baseline_model, img_tensor, PATCH_SIZE, PATCH_OVERLAP, DEVICE)
    pred_attn = sliding_window_inference(attn_model,     img_tensor, PATCH_SIZE, PATCH_OVERLAP, DEVICE)

flair = image[0]
mid_z = gt_seg.shape[2] // 2

gt_bin   = (gt_seg > 0).astype(np.uint8)
bl_bin   = (pred_bl > 0).astype(np.uint8)
attn_bin = (pred_attn > 0).astype(np.uint8)

# Error maps
bl_fp   = (bl_bin == 1) & (gt_bin == 0)    # False positives
bl_fn   = (bl_bin == 0) & (gt_bin == 1)    # False negatives
attn_fp = (attn_bin == 1) & (gt_bin == 0)
attn_fn = (attn_bin == 0) & (gt_bin == 1)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f'Error Analysis: False Positives (FP) & False Negatives (FN)\n{Path(folder).name}',
             fontsize=13, fontweight='bold')

def show_slice(ax, bg, mask, color, title):
    bg_norm = (bg - bg.min()) / (bg.max() - bg.min() + 1e-8)
    ax.imshow(bg_norm.T, cmap='gray', origin='lower')
    m = np.ma.masked_where(mask == 0, mask)
    ax.imshow(m.T, cmap=plt.cm.colors.ListedColormap([color]), origin='lower', alpha=0.7)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

bg = flair[:, :, mid_z]
show_slice(axes[0,0], bg, gt_bin[:,:,mid_z],    '#2ecc71', 'Ground Truth (WT)')
show_slice(axes[0,1], bg, bl_bin[:,:,mid_z],    '#3498db', 'Baseline Prediction')
show_slice(axes[0,2], bg, bl_fp[:,:,mid_z],     '#e74c3c', f'Baseline FP\n({bl_fp.sum():,} voxels)')
show_slice(axes[0,3], bg, bl_fn[:,:,mid_z],     '#f39c12', f'Baseline FN\n({bl_fn.sum():,} voxels)')

show_slice(axes[1,0], bg, gt_bin[:,:,mid_z],    '#2ecc71', 'Ground Truth (WT)')
show_slice(axes[1,1], bg, attn_bin[:,:,mid_z],  '#9b59b6', 'Attn Prediction (Ours)')
show_slice(axes[1,2], bg, attn_fp[:,:,mid_z],   '#e74c3c', f'Attention FP\n({attn_fp.sum():,} voxels)')
show_slice(axes[1,3], bg, attn_fn[:,:,mid_z],   '#f39c12', f'Attention FN\n({attn_fn.sum():,} voxels)')

plt.tight_layout()
plt.savefig('results/figures/error_analysis.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: results/figures/error_analysis.png')
print(f'\nBaseline  — FP: {bl_fp.sum():,} voxels | FN: {bl_fn.sum():,} voxels')
print(f'Attention — FP: {attn_fp.sum():,} voxels | FN: {attn_fn.sum():,} voxels')
print('\n🏁 Notebook 07 complete — all visualisations generated!')
print('📁 Figures saved to: results/figures/')